<a href="https://colab.research.google.com/github/scelliot-cpu/Elliott_DSPN_26_FinalProject/blob/main/Elliott_DSPN_S26_FinalProject.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# How Age and Reaction Time Influence Sensorimotor Adaptation


##Background
Sensorimotor adaptation is the process by which individuals adjust their movements in response to altered sensory feedback. In this dataset, participants completed a visuomotor rotation task where the cursor was rotated relative to hand movement.

This project investigates how individual differences in age and reaction time relate to motor adaptation. Understanding these relationships can provide insight into how learning and motor control change across development.

##Variables of Interest

**Subject ID:** Subject.ID

**Demographic Measures:**

*   **Sex**
*   **Age**

**Trial-Level Task Variables:**

*   TN: Trial Number
*   Block: Task Block
*   ST: Search Time
*   RT: Reaction Time
*   MT: Movement Time
*   ri: Size of Rotation








##Hypotheses
H1: Children will show reduced early motor adaptation relative to young adults.

H2: Children will show reduced late motor adaptation relative to young adults.

H3: Children will show smaller aftereffects than young adults.

H4: Children will have a slower reaction time than young adults.

##Data Plan Organization

###Data Architecture
The raw dataset is trial-level, meaning that each row corresponds to one trial from one participant. Each participant contributes multiple trials across baseline, adaptation, and aftereffect phases.

###Data Cleansing
1.   Define Developmental Groups: Child (<18) and Young Adults (18-30)
2.   Seperate trials into baseline, early adaptation, late adaptation, and aftereffect phases.
3. Aggregate trial-level data into participant-level summary tables for analysis.







In [7]:
install.packages(c("googledrive", "tidyverse", "ggplot2", "tidyr", "GGally", "car"))

library(googledrive)
library(tidyverse)
library(ggplot2)
library(tidyr)
library(GGally)
library(car)

Installing packages into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

also installing the dependencies ‘colorspace’, ‘fracdiff’, ‘lmtest’, ‘timeDate’, ‘urca’, ‘zoo’, ‘RcppArmadillo’, ‘cowplot’, ‘Deriv’, ‘forecast’, ‘microbenchmark’, ‘rbibutils’, ‘patchwork’, ‘numDeriv’, ‘doBy’, ‘SparseM’, ‘MatrixModels’, ‘Rdpack’, ‘minqa’, ‘nloptr’, ‘reformulas’, ‘RcppEigen’, ‘ggstats’, ‘carData’, ‘abind’, ‘Formula’, ‘pbkrtest’, ‘quantreg’, ‘lme4’


── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ forcats   1.0.1     ✔ readr     2.2.0
✔ ggplot2   4.0.2     ✔ stringr   1.6.0
✔ lubridate 1.9.5     ✔ tibble    3.3.1
✔ purrr     1.2.1     ✔ tidyr     1.3.2
── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors
Loading required package: carData


Attaching pa

In [ ]:
file_id <- "1o4Hgv1CyToFkHTmtSoyfyV3r2j9tLsrS"

url <- paste0("https://drive.google.com/uc?id=", file_id)

data <- read.csv(url)

head(data)

In [ ]:
#Define Age Groups
%%R
library(dplyr)

clean_data <- data %>%
  mutate(Group = case_when(
  Age < 18 ~ "Child",
  Age >= 18 & Age <=30 ~ "YoungAdult"
  ))

In [ ]:
#Define Adaptation Phases
clean_data <- clean_data %>%
  mutate(Phase = case_when(
    Block == 1 ~ "Baseline",
    Block == 2 ~ "Adaptation",
    Block == 3 ~ "Aftereffect",
    Block == 4 ~ "OppositeAdaptation",
  ))

clean_data <- clean_data %>%
  group_by(Subject.ID, Phase) %>%
  mutate(Trial_in_Phase = row_number()) %>%
  ungroup()

clean_data <- clean_data %>%
  mutate(Stage = case_when(
    Phase == "Adaptation" & Trial_in_Phase <= 10 ~ "Early Adaptation",
    Phase == "Adaptation" & Trial_in_Phase > 10 ~ "Late Adaptation",
    TRUE ~ Phase
  ))

##Statistical Analysis
This project includes three levels of analysis:

1. **Main Analysis:** compare children and young adults across early adaptation, late adaptation, and aftereffect phases
2. **Secondary Analysis:** compare children and young adults on baseline MT, ST, RT, and hand angle variability
3. **Explatory Analysis:** test whether demographic and behavioral measures predict motor variability in children